# Phase 1: Data Preparation & Filtering

## Overview

This notebook implements Phase 1 of the mechanical engineering job analysis pipeline using the utility modules. The goal is to extract and filter Burning Glass Technologies (BGT) job posting data for the study period (2010-2022) and focus on mechanical engineering positions.

## Methodology

Following Alsharif (2025), we:
1. Query the complete BGT dataset (~422M job postings, 2007-2023)
2. Filter to study period (2010-2022) → ~372M postings
3. Filter to mechanical engineering O*NET codes → ~1.17M postings
4. Assign discipline labels (Civil, Electrical, Mechanical)
5. Analyze missing values
6. Export filtered dataset for Phase 2

## Setup and Configuration

In [ ]:
# Import required libraries
import polars as pl
import numpy as np
import json
from datetime import datetime
from pathlib import Path
import sys

# Add project root to path for imports
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

# Import utility modules
from utils import onet_utils
from utils import DuckDBManager

print("✓ Imports successful")
print(f"Working directory: {project_root}")

In [ ]:
# Load configuration
config_path = project_root / 'config' / 'analysis_config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

phase1_config = config['phase_1']

print("Configuration loaded:")
print(f"  Study period: {phase1_config['study_period']['start']} to {phase1_config['study_period']['end']}")
print(f"  Data source: {phase1_config['data_source']}")
print(f"  Base path: {phase1_config['base_path']}")
print(f"  Output file: {phase1_config['output_file']}")

In [ ]:
# Initialize DuckDB manager for data access
manager = DuckDBManager(
    base_path=phase1_config['base_path'],
    read_only=False,
    data_source=phase1_config['data_source']
)

# Get dataset size
total_size = manager.get_total_parquet_size()
print(f"Total size of Parquet files: {total_size:.2f} MB")

## Data Loading and Baseline Summary

First, we establish a baseline by querying the complete dataset to understand the full scope of available data.

In [ ]:
# Get overall dataset summary
print("=" * 80)
print("BASELINE DATA SUMMARY")
print("=" * 80)

summary = manager.get_data_summary()

print(f"\nTotal job postings: {summary['overview']['total_jobs']:,}")
print(f"Date range: {summary['overview']['date_range']}")
print(f"Unique O*NET codes: {summary['overview']['unique_onet_codes']:,}")
print(f"\nThis represents the complete BGT dataset (2007-2023).")

## O*NET Code Preparation

Following Alsharif (2025) Table 3.1 (page 30), we focus on mechanical engineering O*NET codes:
- **17-2141.00**: Mechanical Engineers (our primary focus)

We use the `onet_utils` module to handle format conversion between:
- **Standard format**: '17-2141.00' (human-readable)
- **Numeric format**: '17214100' (used in BGT Parquet files)

In [ ]:
# Load discipline codes from configuration
discipline_codes = onet_utils.load_discipline_codes()

print("=" * 80)
print("O*NET CODE MAPPING")
print("=" * 80)

for discipline, codes in discipline_codes.items():
    print(f"\n{discipline.capitalize()} Engineering:")
    for std_code, num_code in zip(codes['standard'], codes['numeric']):
        print(f"  {std_code} → {num_code}")

# Extract mechanical engineering codes for querying
mechanical_numeric = discipline_codes['mechanical']['numeric']
print(f"\nMechanical engineering codes for filtering: {mechanical_numeric}")

## Study Period Filtering

We filter the dataset to our study period (2010-2022) for two reasons:
1. **Consistency with literature**: Matches temporal scope of previous studies
2. **Data completeness**: Full years with consistent BGT coverage

This filtering reduces ~422M total postings to ~372M within our timeframe.

In [ ]:
# Count total jobs in study period (before discipline filtering)
study_period_query = f"""
    SELECT COUNT(*) as count
    FROM all_jobs 
    WHERE JobDate BETWEEN '{phase1_config['study_period']['start']}' 
                      AND '{phase1_config['study_period']['end']}'
"""

result = manager.query_with_duckdb(study_period_query)
study_period_total = result['count'].item()

print(f"Jobs in study period (2010-2022): {study_period_total:,}")
print(f"Percentage of complete dataset: {study_period_total/summary['overview']['total_jobs']*100:.1f}%")

## Mechanical Engineering Job Extraction

Now we filter to mechanical engineering positions using O*NET codes. BGT uses O*NET codes to classify occupations, providing a standardized way to identify mechanical engineering jobs.

**Note:** This extraction checks if the output file already exists to avoid re-querying the large dataset unnecessarily.

In [ ]:
print("=" * 80)
print("MECHANICAL ENGINEERING JOB EXTRACTION")
print("=" * 80)

# Check if output file already exists
output_file = f"{phase1_config['output_directory']}/{phase1_config['output_file']}"
file_exists = manager.check_parquet_exists(output_file)

print(f"\nChecking for existing file: {output_file}")
print(f"File exists: {file_exists}")

if not file_exists:
    print("\nQuerying database for mechanical engineering jobs...")
    print("(This may take several minutes for the full dataset)")
    
    # Build query for mechanical engineering jobs in study period
    onet_codes_str = ','.join([f"'{code}'" for code in mechanical_numeric])
    
    mechanical_query = f"""
        SELECT *
        FROM all_jobs 
        WHERE JobDate BETWEEN '{phase1_config['study_period']['start']}' 
                          AND '{phase1_config['study_period']['end']}'
        AND ConsolidatedONET IN ({onet_codes_str})
    """
    
    # Execute query and save to Parquet
    result_path = manager.query_to_parquet(
        mechanical_query,
        phase1_config['output_file']
    )
    
    study_data = pl.read_parquet(result_path)
    print(f"\n✓ Query complete: {len(study_data):,} records extracted")
else:
    print("\nLoading existing data...")
    study_data = pl.read_parquet(manager.base_path / output_file)
    print(f"✓ Loaded {len(study_data):,} records from existing file")

print(f"\nMechanical engineering jobs: {len(study_data):,}")
print(f"Percentage of study period: {len(study_data)/study_period_total*100:.3f}%")

## Discipline Assignment

Although we filtered to mechanical engineering O*NET codes, we assign explicit discipline labels using the `onet_utils` module. This creates a 'Discipline' column that can be used for filtering and analysis.

**Methodology:**
1. Create a lookup dictionary mapping O*NET codes → disciplines
2. Apply the mapping using Polars' vectorized operations
3. Verify that all records are labeled 'Mechanical'

In [ ]:
print("=" * 80)
print("DISCIPLINE ASSIGNMENT")
print("=" * 80)

# Create discipline lookup for vectorized operations
discipline_lookup = onet_utils.create_discipline_lookup_expression(
    discipline_codes,
    format='numeric'
)

print(f"\nCreated lookup for {len(discipline_lookup)} O*NET codes")

# Apply discipline mapping using Polars
study_data = study_data.with_columns(
    pl.col("ConsolidatedONET")
    .replace(discipline_lookup, default="Other")
    .alias("Discipline")
)

# Verify discipline distribution
discipline_counts = study_data.group_by("Discipline").agg(
    pl.len().alias("count")
).sort("count", descending=True)

print("\nDiscipline distribution:")
for row in discipline_counts.iter_rows(named=True):
    percentage = (row['count'] / len(study_data)) * 100
    print(f"  {row['Discipline']}: {row['count']:,} ({percentage:.1f}%)")

# Remove any non-mechanical records (safety check)
study_data = study_data.filter(pl.col("Discipline") == "Mechanical")
print(f"\n✓ Final mechanical engineering records: {len(study_data):,}")

## Temporal Coverage Verification

We verify that our filtered dataset covers the complete study period (2010-2022) and extract the year for later analysis.

In [ ]:
# Add Year column for analysis
study_data = study_data.with_columns(
    pl.col("JobDate").str.strptime(pl.Date, "%Y-%m-%d").dt.year().alias("Year")
)

# Get temporal range
year_stats = study_data.select([
    pl.col("Year").min().alias("min_year"),
    pl.col("Year").max().alias("max_year"),
    pl.col("JobDate").max().alias("last_date")
]).to_pandas().iloc[0]

print("\nTemporal Coverage:")
print(f"  First year: {year_stats['min_year']}")
print(f"  Last year: {year_stats['max_year']}")
print(f"  Latest date: {year_stats['last_date']}")
print(f"  Years covered: {year_stats['max_year'] - year_stats['min_year'] + 1}")

## Missing Values Analysis

Following Alsharif (2025) Table 3.3, we analyze missing values in key fields. This helps us understand data quality and determine which records can be used for different analyses.

**Key fields checked:**
- **JobID**: Unique identifier (should never be missing)
- **CleanJobTitle**: Standardized job title (occasionally missing)
- **JobText**: Full job description (critical for NLP classification)
- **ConsolidatedONET**: O*NET occupation code (used for filtering)
- **JobDate**: Posting date (used for temporal analysis)
- **CanonSkillClusters**: BGT-extracted skills (used in Phase 3)

In [ ]:
print("=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)

# Fields to check for missing values
fields_to_check = [
    'JobID',
    'CleanJobTitle',
    'JobText',
    'ConsolidatedONET',
    'JobDate',
    'CanonSkillClusters'
]

total_records = len(study_data)

print(f"\nTotal records analyzed: {total_records:,}")
print("\n{:<25} {:>12} {:>12} {:>12} {:>12}".format(
    "Field", "Missing", "Missing %", "Present", "Present %"
))
print("-" * 80)

for field in fields_to_check:
    if field in study_data.columns:
        missing_count = study_data.select(pl.col(field).is_null().sum()).item()
        missing_pct = (missing_count / total_records) * 100
        present_count = total_records - missing_count
        present_pct = 100 - missing_pct
        
        print("{:<25} {:>12,} {:>11.2f}% {:>12,} {:>11.2f}%".format(
            field, missing_count, missing_pct, present_count, present_pct
        ))

print("-" * 80)

# Identify records with missing CanonSkillClusters (impacts Phase 3)
missing_skills = study_data.filter(pl.col("CanonSkillClusters").is_null()).height
print(f"\n⚠️  Records missing skill clusters: {missing_skills:,} ({missing_skills/total_records*100:.2f}%)")
print("   These records will be excluded from Phase 3 skills analysis.")

## Data Reduction Summary

We track how the dataset size changes through each filtering step. This provides transparency about what proportion of data is retained and excluded at each stage.

In [ ]:
print("=" * 80)
print("DATA REDUCTION SUMMARY")
print("=" * 80)

# Create summary DataFrame
reduction_summary = pl.DataFrame({
    'Stage': [
        'Full Dataset (2007-2023)',
        'Study Period (2010-2022)',
        'Mechanical Engineering Only',
        'With Complete Skill Data'
    ],
    'Record_Count': [
        summary['overview']['total_jobs'],
        study_period_total,
        len(study_data),
        len(study_data) - missing_skills
    ]
})

# Calculate retention percentage
reduction_summary = reduction_summary.with_columns(
    ((pl.col("Record_Count") / pl.col("Record_Count").first()) * 100)
    .round(2)
    .alias("Retention_%")
)

print("\n" + str(reduction_summary))

# Calculate reduction ratios
me_retention = len(study_data) / study_period_total * 100
print(f"\n📊 Key Statistics:")
print(f"   • Mechanical engineering jobs represent {me_retention:.3f}% of study period")
print(f"   • {len(study_data):,} mechanical engineering jobs retained")
print(f"   • {missing_skills:,} jobs lack skill cluster data ({missing_skills/len(study_data)*100:.2f}%)")

## Export Filtered Dataset

Finally, we export the filtered dataset to Parquet format for use in Phase 2 (job classification). The Parquet format is chosen for:
- **Efficiency**: Compressed columnar storage
- **Speed**: Fast read/write operations
- **Compatibility**: Works with Polars, Pandas, DuckDB

In [ ]:
print("=" * 80)
print("EXPORTING FILTERED DATASET")
print("=" * 80)

# Export to Parquet
output_path = Path(phase1_config['base_path']) / output_file
study_data.write_parquet(output_path)

print(f"\n✓ Dataset exported successfully")
print(f"  Path: {output_path}")
print(f"  Records: {len(study_data):,}")
print(f"  Columns: {len(study_data.columns)}")
print(f"  File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

# List key columns for reference
print(f"\n📋 Key columns in output:")
key_columns = ['JobID', 'CleanJobTitle', 'JobText', 'Discipline', 'ConsolidatedONET', 
               'CanonSkillClusters', 'JobDate', 'Year']
for col in key_columns:
    if col in study_data.columns:
        print(f"   • {col}")

## Phase 1 Complete ✓

### Summary

We successfully prepared the mechanical engineering job dataset for analysis:

**Data Pipeline:**
1. ✓ Loaded BGT dataset (~422M jobs, 2007-2023)
2. ✓ Filtered to study period (2010-2022) → ~372M jobs
3. ✓ Extracted mechanical engineering jobs → ~1.17M jobs
4. ✓ Assigned discipline labels using O*NET codes
5. ✓ Analyzed missing values (0.84% missing skill data)
6. ✓ Exported filtered dataset for Phase 2

**Output:**
- File: `study_data_ME_ONLY.parquet`
- Records: ~1,166,271 mechanical engineering jobs
- Time period: 2010-2022 (13 years)
- Ready for Phase 2 NLP classification

### Next Steps

Proceed to **Phase 2** (`phase_2.ipynb`) for:
- NLP-based job classification
- Confidence scoring with Bayesian-optimized parameters
- Adaptive thresholding by job category
- Identification of "major" mechanical engineering positions (~84% of dataset)

### References

- Alsharif, M. (2025). Engineering Employment Trends Study. [Dissertation]
- O*NET occupation codes: Table 3.1, page 30
- Missing values analysis: Table 3.3